# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahsan-Qadeer/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!pip -q install duckdb pyarrow huggingface_hub

#Login using token
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

In [15]:
#Connecting to the warehouse
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET (TYPE huggingface,TOKEN '{userdata.get("HF_TOKEN")}')""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

## 1. Unit of analysis + time window

One row represents one content page for one client, respectively on one reporting date from fact_content_daily_performance table.
I am going to be using the March 2026 partition as development month since it is the mid panel month, not the final outcome month.

In [16]:
example = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

example

,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72


## 2. Fields: feature / label / context / excluded

###Features used by the model
gsc_impressions: measure search visibility

gsc_clicks: user engagement from search

ctr: click through rate

gsc_avg_position: average Google search ranking

days_since_last_update: shows how stale content is



###Label/Proxy
Declining content

This is the outcome the model is going to try to rank or predict

###Context
client_hash_id: identifies the client

content_hash_id: identifies the content page

report_date: shows the observatiion date

###Excluded
Label derived fields like future trend or decline indicators. This is because they leak the answer to the model

Future months since they are reserved as outcome or test period

Any client identifying information, since  dataset is pseudonymized and the task needs to stay privacy preserving


In [20]:
fields = con.sql(f"""
SELECT gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, report_date, client_hash_id, content_hash_id
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

fields

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,report_date,client_hash_id,content_hash_id
0,20,0,3.350000,<NA>,<NA>,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,1,0,0.000000,<NA>,<NA>,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067
2,125,1,4.928000,<NA>,<NA>,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,7,0,4.000000,<NA>,<NA>,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,11,0,2.272727,<NA>,<NA>,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72


## 3. Verify it with queries (grain, counts, missing values, windows)

I have verified three parts of the data contract:

*The grain is one row per client, content item and report date*

*The March 2026 partition has a measurable row count and date window*

*I confirmed how many rows have Google search Console data available*

In [18]:
#Verifying the grain (one row = one client, one content item, one report date)

grain = con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

print("\nGrain")
print(grain)


#Confirming row count and date window

counts = con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS first_date, MAX(report_date) AS last_date
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("\nRow count and date span")
print(counts)


#Verifying availability

availability = con.sql(f"""
SELECT COUNT(*) AS rows_with_gsc_data
FROM read_parquet('{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
""").df()

print("\nGSC data available")
print(availability)


Grain
            client_hash_id           content_hash_id report_date
0  client_73cda7b4e4f265ea  content_b7e512995f79d5a6  2026-03-01
1  client_73cda7b4e4f265ea  content_05597932fe4da067  2026-03-01
2  client_73cda7b4e4f265ea  content_7a105f548d9c6916  2026-03-01
3  client_73cda7b4e4f265ea  content_905aa32a0230694e  2026-03-01
4  client_73cda7b4e4f265ea  content_a3ea9792f793ec72  2026-03-01


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Row count and date span
   row_count first_date  last_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


GSC data available
   rows_with_gsc_data
0             3611061


## 4. Data limits

The dataset has a few limitations, some of which are listed below:

Different clients began giving data at different times. Historical coverage is thus not equal for all clients

Some clients have onlly GSC data or only GA4 data, so not every metric is available for every row

The dataset shows what happeneed but cant prove that refreshing content resulted in changes in performance

Metrics calculated over rolling time windows mighht ovoerlap, so observations close to one another are not fully independent

Client and content identifiers are anonymized, which prevents linking data back to real business context

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.